In [ ]:
# Cell 1: Cài đặt thư viện
!pip install web3 py-solc-x gradio -q
print("✅ Đã cài đặt xong thư viện.")

✅ Đã cài đặt xong thư viện.


In [ ]:
# --- CELL 1: KẾT NỐI VÍ BUYER ---
import json, os
from web3 import Web3
from google.colab import files
import gradio as gr

# 1. Cấu hình (Thay URL Infura và Private Key của bạn nếu cần)
INFURA_URL = "https://sepolia.infura.io/v3/ce4c6a48759f46f9bc789a3912fbc0f6"
SHOP_ADDRESS = "0x47D821A5B2a86bBECeb07bd05E1Ed32C196E732e"
BUYER_KEY = "7f2ebc3b8bb5d73e320c6c38b980c0c1198deee97b7097b74b4b12aa0569ce3e"

# 2. Kết nối
w3 = Web3(Web3.HTTPProvider(INFURA_URL))
buyer = w3.eth.account.from_key(BUYER_KEY)

# 3. Load Contract
if not os.path.exists("ShopABI.json"):
    print("📂 Hãy upload file ShopABI.json...")
    files.upload()

with open("ShopABI.json") as f: abi = json.load(f)
contract = w3.eth.contract(address=w3.to_checksum_address(SHOP_ADDRESS), abi=abi)

print(f"✅ Buyer đã sẵn sàng: {buyer.address}")
print(f"💰 Số dư hiện tại: {w3.from_wei(w3.eth.get_balance(buyer.address), 'ether')} ETH")

✅ Buyer đã sẵn sàng: 0x2F2DCF2F3682FA81a20A3E45323f287BB669F9E9
💰 Số dư hiện tại: 2.413090326909279604 ETH


In [ ]:
# --- CELL 2: LOGIC XỬ LÝ (CẬP NHẬT) ---

# 1. Xem danh sách hàng
def list_products():
    res = []
    try:
        c = contract.functions.productCounter().call()
        for i in range(c):
            p = contract.functions.products(i).call()
            if p[3] > 0: # Còn hàng
                price = Web3.from_wei(p[2], 'ether')
                res.append(f"🆔 {p[0]} | 📦 {p[1]} | 💰 {price} ETH | 🔢 Kho: {p[3]}")
        return "\n".join(res) if res else "📭 Chưa có sản phẩm."
    except Exception as e: return f"❌ Lỗi: {e}"

# 2. Tính giá
def calculate_total(pid, qty):
    try:
        p = contract.functions.products(int(pid)).call()
        total = w3.from_wei(p[2] * int(qty), 'ether')
        return f"🛒 Mua: {p[1]} (x{qty})\n💰 Cần trả: {total} ETH"
    except: return "❌ Kiểm tra lại ID và Số lượng"

# 3. Mua hàng
def buy_now(pid, qty):
    try:
        p = contract.functions.products(int(pid)).call()
        val = p[2] * int(qty)
        print("⏳ Đang thanh toán...")
        tx = contract.functions.buyProduct(int(pid), int(qty)).build_transaction({
            'from': buyer.address, 'value': val,
            'nonce': w3.eth.get_transaction_count(buyer.address),
            'gas': 2000000, 'gasPrice': w3.eth.gas_price
        })
        signed = w3.eth.account.sign_transaction(tx, buyer.key)
        w3.eth.wait_for_transaction_receipt(w3.eth.send_raw_transaction(signed.raw_transaction))
        return "✅ Mua thành công! Kiểm tra tab Lịch Sử."
    except Exception as e: return f"❌ Lỗi: {e}"

# 4. Tra cứu đơn (Có trạng thái)
def track_order(oid):
    try:
        if not oid: return "⚠️ Nhập ID đơn hàng!"
        o = contract.functions.orders(int(oid)).call()
        if o[2] != buyer.address: return "⛔ Không phải đơn của bạn!"
        p = contract.functions.products(o[1]).call()
        st = ["🟢 Chờ Ship", "🚚 Đang giao", "✅ Hoàn tất", "⚖️ Khiếu nại", "↩️ Hoàn tiền"][o[5]]
        return f"📦 Đơn #{oid} | {p[1]} (x{o[3]}) | {st}"
    except: return "❌ Không tìm thấy đơn."

# 5. Xác nhận / Khiếu nại
def process_order(oid, action, reason):
    try:
        if "Ok" in action: fn = contract.functions.confirmReceipt(int(oid))
        else: fn = contract.functions.raiseDispute(int(oid), reason)
        tx = fn.build_transaction({
            'from': buyer.address, 'nonce': w3.eth.get_transaction_count(buyer.address),
            'gas': 500000, 'gasPrice': w3.eth.gas_price
        })
        signed = w3.eth.account.sign_transaction(tx, buyer.key)
        w3.eth.wait_for_transaction_receipt(w3.eth.send_raw_transaction(signed.raw_transaction))
        return "✅ Thành công!"
    except Exception as e: return f"❌ Lỗi: {e}"

# 6. Sao kê tiền (Giữ nguyên)
def view_transaction_statement():
    lines = []
    balance_change = 0
    try:
        count = contract.functions.orderCounter().call()
        lines.append(f"{'ĐƠN HÀNG':<10} | {'LOẠI GD':<20} | {'SỐ TIỀN':<15}")
        lines.append("-" * 60)
        for i in range(count):
            o = contract.functions.orders(i).call()
            if o[2] == buyer.address:
                amt = float(w3.from_wei(o[4], 'ether'))
                lines.append(f"#{o[0]:<9} | 🛒 Mua hàng           | -{amt:<14.5f}")
                balance_change -= amt
                if o[5] == 4:
                    lines.append(f"#{o[0]:<9} | ↩️ Hoàn tiền          | +{amt:<14.5f}")
                    balance_change += amt
        lines.append("-" * 60)
        lines.append(f"TỔNG CHI TIÊU THỰC TẾ: {balance_change:+.5f} ETH")
        return "\n".join(lines)
    except Exception as e: return f"❌ Lỗi: {e}"

# 7. (MỚI) LỊCH SỬ ĐƠN HÀNG (KHÔNG HIỆN TRẠNG THÁI)
def view_simple_order_history():
    lines = []
    try:
        count = contract.functions.orderCounter().call()

        # Header đơn giản
        header = f"{'MÃ ĐƠN':<8} | {'TÊN SẢN PHẨM':<30} | {'SL':<5} | {'TỔNG TIỀN'}"
        lines.append(header)
        lines.append("-" * 70)

        for i in range(count):
            o = contract.functions.orders(i).call()
            # Chỉ lấy đơn của mình (buyer)
            if o[2] == buyer.address:
                p = contract.functions.products(o[1]).call()
                amount_eth = float(w3.from_wei(o[4], 'ether'))

                # Format dòng tin: ID | Tên SP | Số lượng | Giá
                # Tuyệt đối KHÔNG lấy o[5] (trạng thái)
                line = f"#{o[0]:<7} | {p[1]:<30} | x{o[3]:<4} | {amount_eth} ETH"
                lines.append(line)

        if len(lines) <= 2: return "Bạn chưa đặt đơn hàng nào."
        return "\n".join(lines)

    except Exception as e: return f"❌ Lỗi: {e}"

In [ ]:
# --- CELL 3: GIAO DIỆN NGƯỜI MUA (UI ĐẸP) ---
import gradio as gr

# Sử dụng theme Soft màu xanh ngọc (Teal) cho hiện đại
my_theme = gr.themes.Soft(
    primary_hue="teal",
    secondary_hue="blue",
    text_size="lg",
    spacing_size="md",
).set(
    button_primary_background_fill="*primary_500",
    button_primary_background_fill_hover="*primary_600",
)

with gr.Blocks(title="Blockchain Mall", theme=my_theme) as app:

    # Header chung
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("# 🛍️ BUYER MALL\n### Sàn thương mại điện tử phi tập trung")

    # --- TAB 1: MUA SẮM ---
    with gr.Tab("🛒 Mua Sắm & Thanh Toán"):
        with gr.Row():
            # Cột Trái: Hiển thị sản phẩm
            with gr.Column(scale=3):
                gr.Markdown("### 📋 Danh Sách Sản Phẩm")
                btn_refresh = gr.Button("🔄 Cập nhật danh sách", variant="secondary")
                out_list = gr.Textbox(label="Sản phẩm đang bán", lines=15, show_copy_button=True)
                btn_refresh.click(list_products, outputs=out_list)

            # Cột Phải: Form Mua Hàng
            with gr.Column(scale=2, variant="panel"):
                gr.Markdown("### 💳 Giỏ Hàng")
                with gr.Group():
                    pid = gr.Textbox(label="🆔 Nhập ID Sản Phẩm", placeholder="Ví dụ: 0")
                    qty = gr.Textbox(label="🔢 Số Lượng", placeholder="Ví dụ: 1")

                # Nút tính giá
                btn_calc = gr.Button("🧮 Xem Tạm Tính")
                out_calc = gr.Textbox(label="Hóa Đơn Tạm Tính", lines=3)

                # Nút Mua (Nổi bật)
                btn_buy = gr.Button("🔥 ĐẶT MUA NGAY", variant="primary", size="lg")
                out_buy = gr.Textbox(label="Trạng thái giao dịch")

                # Sự kiện
                btn_calc.click(calculate_total, [pid, qty], out_calc)
                btn_buy.click(buy_now, [pid, qty], out_buy)

    # --- TAB 2: QUẢN LÝ ĐƠN HÀNG ---
    with gr.Tab("📦 Quản Lý Đơn Hàng"):
        gr.Markdown("Theo dõi trạng thái đơn hàng và xác nhận khi nhận được hàng.")

        with gr.Row():
            # Khung Tra Cứu
            with gr.Column(variant="panel"):
                gr.Markdown("### 🔍 Tra Cứu Vận Đơn")
                check_id = gr.Textbox(label="Nhập Mã Đơn Hàng (Order ID)", placeholder="Nhập ID để kiểm tra...")
                btn_check = gr.Button("Kiểm Tra Trạng Thái", variant="secondary")
                out_check = gr.Textbox(label="Chi Tiết Đơn Hàng", lines=5)

                btn_check.click(track_order, check_id, out_check)

            # Khung Xử Lý
            with gr.Column(variant="panel"):
                gr.Markdown("### ✅ Xác Nhận Nhận Hàng")
                gr.Markdown("_Chỉ thực hiện khi bạn đã nhận được hàng hoặc muốn khiếu nại._")

                with gr.Group():
                    oid = gr.Textbox(label="Mã Đơn Hàng")
                    act = gr.Dropdown(
                        ["✅ Đã nhận hàng (Ok)", "🔥 Khiếu nại (Lỗi)"],
                        label="Hành động",
                        value="✅ Đã nhận hàng (Ok)"
                    )
                    rs = gr.Textbox(label="Lý do (Nếu chọn Khiếu nại)")

                btn_process = gr.Button("Gửi Xác Nhận", variant="stop")
                out_process = gr.Textbox(label="Kết quả xử lý")

                btn_process.click(process_order, [oid, act, rs], out_process)

    # --- TAB 3: LỊCH SỬ & TÀI CHÍNH ---
    with gr.Tab("📒 Lịch Sử & Ví"):
        with gr.Row():
            # Cột Lịch Sử
            with gr.Column():
                gr.Markdown("### 📜 Lịch Sử Đã Đặt")
                btn_hist = gr.Button("Xem danh sách đơn (Rút gọn)")
                out_hist = gr.Textbox(label="Nhật ký đặt hàng", lines=12)
                btn_hist.click(view_simple_order_history, outputs=out_hist)

            # Cột Tài Chính
            with gr.Column():
                gr.Markdown("### 💰 Biến Động Số Dư")
                btn_stat = gr.Button("Xem Sao Kê Chi Tiết")
                out_stat = gr.Textbox(label="Dòng tiền ra/vào", lines=12)
                btn_stat.click(view_transaction_statement, outputs=out_stat)

# Chạy ứng dụng
app.launch(share=True)

/tmp/ipython-input-3555530409.py:15: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Blockchain Mall", theme=my_theme) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fe0e586c4bc4ae6dd7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
